# Combat balance framework

This notebook explains the combat damage formula and explores its sensitivity to progression, stat allocation and effect power. It evaluates controlled scenarios, not win rates or a complete combat simulation.

The six damage-related stats are **Offence, Defence, Strength, Speed, Intelligence and Willpower**. Masteries unlock content and do not enter the damage formula. Experience is independent of the stat total; use the effective battle level and battle-state stats as inputs.

Repository sources:
- `app/src/libs/combat/tags.ts`: `damageCalc`, `getPower`, `getLowerGenerals`.
- `app/src/libs/combat/constants.ts`: default damage configuration.
- `app/src/libs/combat/process.ts`: `computeDamagePacket` and modifier ordering.
- `app/src/libs/profile.ts`: HP and Energy capacities.
- `app/src/server/api/routers/train.ts`: instant Energy spending and timed mastery training.
- `app/drizzle/constants.ts`: tuning values and rank caps.

Run all cells from the repository root or its `scripts` directory. Requirements: Python with NumPy, pandas and Matplotlib; Bun and installed application dependencies. Configuration is imported from the repository on each run. These are code defaults; live game settings can override damage tuning. Supply those overrides explicitly to analyze a configured environment.


In [ ]:
import json
import math
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "app/drizzle/constants.ts").exists())
config_source = """
import * as c from './drizzle/constants';
import { dmgConfig } from './src/libs/combat/constants';
console.log(JSON.stringify({
  damage: dmgConfig,
  hp_per_level: c.HP_PER_LVL, energy_per_level: c.ENERGY_PER_LVL,
  stats_per_energy: c.STATS_PER_ENERGY, regen_seconds: c.REGEN_SECONDS,
  base_increase: c.OUT_OF_COMBAT_BASE_DAMAGE_INCREASE,
  base_reduction: c.OUT_OF_COMBAT_BASE_DAMAGE_REDUCTION,
  reduction_cap: c.DMG_REDUCTION_CAP,
  caps: c.USER_CAPS,
  energy_rewards: {quest: c.ENERGY_PVE_REWARD, event: c.ENERGY_EVENT_REWARD,
                   pvp_win: c.ENERGY_PVP_WIN_REWARD, pvp_loss: c.ENERGY_PVP_LOSS_REWARD}
}));
"""
settings = json.loads(subprocess.run(
    ["bun", "-e", config_source], cwd=repo / "app",
    capture_output=True, text=True, check=True,
).stdout)
config = settings["damage"].copy()
plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True})
display(pd.Series(config, name="Default damage configuration").to_frame())


## Formula and inputs

Effect power is $E = \text{power} + \text{level}\times\text{powerPerLevel}$, where `level` is the effect's level. Percentage effects cap this value at 100.

A nonempty `statTypes` list contributes **one** shared Offence/Defence term, regardless of how many classifications it names. An empty list contributes none. Let $s$ be 1 for a nonempty list and 0 otherwise, and $G$ be the effect's resolved general list:

$$A = w_s\,s\sqrt{\max(0,\mathrm{Offence}_{attacker})}
      + w_g\sum_{g\in G}\sqrt{\max(0,g_{attacker})}$$
$$D = w_s\,s\sqrt{\max(0,\mathrm{Defence}_{defender})}
      + w_g\sum_{g\in G}\sqrt{\max(0,g_{defender})}$$

Explicit generals apply to both sides. `Highest` expands to the **attacker's battle-state general selection** on both sides, including preferred generals selected at battle initiation. It does not independently select the defender's strongest generals. Explicit generals and the `Highest` expansion are concatenated; repeated entries contribute repeatedly, as in the engine.

$$H(L) = 100 + h(L-1)$$
$$M = \begin{cases}
1,& A=D=0\\
\operatorname{clip}\left(1+a\left[\left(\frac{A}{\max(1,D)}\right)^c-1\right],m_{min},m_{max}\right),&\text{otherwise}
\end{cases}$$
$$\mathrm{raw\ damage} = \frac{H(L_{attacker})}{\max(1,\mathrm{base\_hits})}
                         \frac{E}{\max(1,\mathrm{ep\_normalization})}\,M$$

The world scale uses the **attacker's effective level and base HP formula**, not actual buffed HP or the defender's level. A residual modifier multiplies damage when the effect was not cast this round; a damage modifier applies afterwards. Zero-valued modifiers are ignored by `damageCalc`. Non-formula effects, or effects without an origin, return effective power before those modifiers; their downstream interpretation depends on the calling effect handler.


In [ ]:
GENERALS = ("strength", "speed", "intelligence", "willpower")

def combatant(level=100, offence=100_000, defence=100_000, general=100_000, **overrides):
    return {"level": level, "offence": offence, "defence": defence,
            **dict.fromkeys(GENERALS, general),
            "highestGenerals": ["intelligence", "willpower"], **overrides}

STANDARD_EFFECT = {"power": config["ep_normalization"], "powerPerLevel": 0,
                   "level": 0, "calculation": "formula", "castThisRound": True,
                   "statTypes": ["Ninjutsu"],
                   "generalTypes": ["Intelligence", "Willpower"]}

def raw_damage(attacker, defender, effect=STANDARD_EFFECT, tuning=config):
    power = effect["power"] + effect["level"] * effect["powerPerLevel"]
    if effect["calculation"] == "percentage":
        power = min(100, power)
    damage = power
    if effect["calculation"] == "formula" and attacker is not None:
        selected = [g.lower() for g in effect.get("generalTypes", []) if g != "Highest"]
        if "Highest" in effect.get("generalTypes", []):
            selected += attacker.get("highestGenerals", [])
        attack = defence = 0.0
        if effect.get("statTypes"):
            attack = tuning["stats_scaling"] * math.sqrt(max(0, attacker["offence"]))
            defence = tuning["stats_scaling"] * math.sqrt(max(0, defender["defence"]))
        for general in selected:
            if general in attacker and general in defender:
                attack += tuning["gen_weight"] * math.sqrt(max(0, attacker[general]))
                defence += tuning["gen_weight"] * math.sqrt(max(0, defender[general]))
        advantage = 1.0
        if attack > 0 or defence > 0:
            ratio = attack / max(1, defence)
            advantage = np.clip(1 + tuning["amplitude"] * (ratio ** tuning["curve"] - 1),
                                tuning["advantage_min"], tuning["advantage_max"])
        base_hp = 100 + settings["hp_per_level"] * (attacker["level"] - 1)
        damage = (base_hp / max(1, tuning["base_hits"]) *
                  power / max(1, tuning["ep_normalization"]) * advantage)
    if not effect.get("castThisRound", False) and effect.get("residualModifier"):
        damage *= effect["residualModifier"]
    if effect.get("dmgModifier"):
        damage *= effect["dmgModifier"]
    return float(damage)

# Neutral packet: no gear, bloodline, skills or active damage modifiers.
neutral_multiplier = ((1 + settings["base_increase"] / 100) *
                      (1 - settings["base_reduction"] / 100))
display(pd.Series({"neutral_packet_multiplier": neutral_multiplier,
                   "raw_standard_hit_at_level_100": raw_damage(combatant(), combatant()),
                   "neutral_standard_hit_at_level_100":
                       raw_damage(combatant(), combatant()) * neutral_multiplier}).to_frame("Value"))


## Agreement with the combat engine

The following scenarios compare the Python model to the actual TypeScript `damageCalc`. They also check the neutral modifier result against `computeDamagePacket`. These are local, pure calculations: no database queries or player changes are made.

Coverage includes classifications, general selection, masteries, effective power, asymmetric levels, zero/negative stats, bounded advantage, residual modifiers, zero modifiers, and non-formula effects. Assertions intentionally fail if the executable model diverges from the engine.


In [ ]:
cases = []
for label, attacker, defender, changes in [
    ("balanced", combatant(), combatant(), {}),
    ("multiple classifications", combatant(), combatant(), {"statTypes": ["Ninjutsu", "Taijutsu"]}),
    ("generals only", combatant(), combatant(), {"statTypes": []}),
    ("stats only", combatant(), combatant(), {"generalTypes": []}),
    ("statless", combatant(), combatant(), {"statTypes": [], "generalTypes": []}),
    ("highest selection", combatant(strength=400_000, intelligence=10), combatant(speed=10), {"generalTypes": ["Highest"]}),
    ("repeated generals", combatant(intelligence=400_000), combatant(), {"generalTypes": ["Intelligence", "Highest"]}),
    ("asymmetric levels", combatant(level=50), combatant(level=100), {}),
    ("masteries", combatant(ninjutsuMastery=1_500_000), combatant(ninjutsuMastery=10), {}),
    ("no origin", None, combatant(), {}),
    ("static effect", combatant(), combatant(), {"calculation": "static"}),
    ("percentage ceiling", combatant(), combatant(), {"calculation": "percentage", "power": 80, "powerPerLevel": 5, "level": 10}),
    ("effect leveling", combatant(), combatant(), {"power": 28, "powerPerLevel": 0.4, "level": 25}),
    ("residual and modifier", combatant(), combatant(), {"castThisRound": False, "residualModifier": 0.5, "dmgModifier": 1.2}),
    ("zero modifiers", combatant(), combatant(), {"castThisRound": False, "residualModifier": 0, "dmgModifier": 0}),
    ("negative stats", combatant(offence=-10, general=-10), combatant(defence=-10, general=-10), {}),
    ("advantage maximum", combatant(offence=1_300_000, general=400_000), combatant(defence=0, general=0), {}),
    ("zero attack", combatant(offence=0, general=0), combatant(), {}),
]:
    cases.append({"label": label, "attacker": attacker, "defender": defender,
                  "effect": {**STANDARD_EFFECT, **changes}})

reference_source = """
import { damageCalc } from './src/libs/combat/tags';
import { computeDamagePacket } from './src/libs/combat/process';
const {cases, config} = JSON.parse(process.argv.at(-1));
console.log(JSON.stringify(cases.map(({label, attacker, defender, effect}) => {
  const raw = damageCalc(effect, attacker ?? undefined, defender, config);
  const packet = computeDamagePacket({rawDamage: raw, damageEffect: effect,
    usersEffects: [], attackerId: 'attacker', defenderId: 'defender',
    preBattleGearModifiers: {}, battleRound: 1}).damage;
  return {label, raw, packet};
})));
"""
reference = json.loads(subprocess.run(
    ["bun", "-e", reference_source, json.dumps({"cases": cases, "config": config})],
    cwd=repo / "app", capture_output=True, text=True, check=True,
).stdout)
for case, actual in zip(cases, reference):
    calculated = raw_damage(case["attacker"], case["defender"], case["effect"])
    np.testing.assert_allclose(calculated, actual["raw"], rtol=1e-12, atol=1e-12)
    np.testing.assert_allclose(calculated * neutral_multiplier, actual["packet"], rtol=1e-12, atol=1e-12)
display(pd.DataFrame(reference).set_index("label").round(4))


## Level, effect power and hit count

With equal nonzero attack/defence power, the advantage modifier is 1. The raw normalized hit is base HP divided by the configured hit target. Default damage increase and reduction apply afterwards, so the raw hit target is not the final number of attacks to defeat an opponent.

The charts use equal attacker/defender levels and base HP, with no pool bonuses or healing. `HP / damage` is a continuous equivalent hit count; round upward for discrete identical hits. It is **not** a round count: action costs, multiple targets, criticals, regeneration, barriers, absorption, reflection, residual ticks and combat decisions change encounters.


In [ ]:
levels = np.arange(1, 101)
hp = 100 + settings["hp_per_level"] * (levels - 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for power in [20, 40, 60]:
    effect = {**STANDARD_EFFECT, "power": power}
    raw = np.array([raw_damage(combatant(level=int(level)), combatant(level=int(level)), effect)
                    for level in levels])
    axes[0].plot(levels, raw * neutral_multiplier, label=f"Power {power}: neutral packet")
    axes[1].plot(levels, hp / (raw * neutral_multiplier), label=f"Power {power}")
axes[0].plot(levels, hp / max(1, config["base_hits"]), "--", color="black", label="Raw normalized hit")
axes[0].set(xlabel="Effective battle level", ylabel="Damage", title="World scale and effect power")
axes[1].set(xlabel="Effective battle level", ylabel="Equivalent hits to defeat base HP", title="Identical hits, equal power on both sides")
for ax in axes: ax.legend()
plt.tight_layout()
plt.show()


## Stat advantage and general sensitivity

Square roots give diminishing power gains per invested point. Selected generals each receive `gen_weight`, while shared Offence/Defence receives `stats_scaling`. A technique with two selected generals sums both contributions. Increasing a general not selected by the effect has no direct effect on that hit.

A shared combat stat applies across classifications, but generals can still create technique-specific matchups. A fixed relative ratio of all relevant stats produces a fixed advantage at any absolute stat scale. Unrelated stats, XP and masteries do not directly change this ratio.


In [ ]:
ratios = np.geomspace(0.001, 100, 500)
modifiers = np.clip(1 + config["amplitude"] * (ratios ** config["curve"] - 1),
                    config["advantage_min"], config["advantage_max"])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].semilogx(ratios, modifiers)
axes[0].axvline(1, color="gray", linestyle="--")
axes[0].set(xlabel="Attack power / max(1, defence power)", ylabel="Advantage multiplier", title="Bounded power advantage")

values = np.linspace(10, settings["caps"]["JONIN"]["GENS_CAP"], 160)
defender = combatant()
for field, label in [("intelligence", "Selected Intelligence"), ("strength", "Unselected Strength")]:
    damage = [raw_damage(combatant(**{field: float(value)}), defender) * neutral_multiplier for value in values]
    axes[1].plot(values, damage, label=label)
axes[1].set(xlabel="General stat value (other inputs fixed)", ylabel="Neutral packet damage", title="Only resolved generals enter the hit")
axes[1].legend()
plt.tight_layout()
plt.show()

cap = settings["caps"]["JONIN"]
stat_term = config["stats_scaling"] * math.sqrt(cap["STATS_CAP"])
general_term = config["gen_weight"] * 2 * math.sqrt(cap["GENS_CAP"])
display(pd.Series({"Shared stat contribution at base cap": stat_term,
                   "Two selected generals at base cap": general_term,
                   "General share of total power": general_term / (stat_term + general_term)}).to_frame("Value"))


## Equal-budget allocations

These scenarios distribute the same number of trained points above a base of 10 across the six combat stats. They isolate allocation rather than equating XP with stats. Offence-heavy allocation favors outgoing damage; Defence-heavy allocation favors surviving incoming hits; selected generals can influence both directions. The model does not assign a universal winning build.

The budget is a scenario input within the displayed rank's base caps. All values passed to the formula are effective battle-state values. Stored overflow is not automatically usable: rank/global caps are applied when battle state is prepared, and equipment or other effects can then alter that state. Mastery caps and bonuses govern eligibility separately.


In [ ]:
rank_caps = pd.DataFrame(settings["caps"]).T
rank_caps.index.name = "Rank"
display(rank_caps)

budget = 600_000
allocations = {
    "Balanced": dict.fromkeys(["offence", "defence", *GENERALS], 1 / 6),
    "Offence-heavy": {"offence": 0.6, "defence": 0.1, "intelligence": 0.15, "willpower": 0.15},
    "Defence-heavy": {"offence": 0.1, "defence": 0.6, "intelligence": 0.15, "willpower": 0.15},
    "Selected generals": {"offence": 0.1, "defence": 0.1, "intelligence": 0.4, "willpower": 0.4},
}
builds = {name: combatant(general=10, offence=10, defence=10,
                         **{field: 10 + budget * share for field, share in shares.items() if field not in ("offence", "defence")})
          for name, shares in allocations.items()}
for name, shares in allocations.items():
    for field in ["offence", "defence"]:
        builds[name][field] = 10 + budget * shares.get(field, 0)
for user in builds.values():
    assert np.isclose(sum(user[field] - 10 for field in ["offence", "defence", *GENERALS]), budget)
    assert max(user["offence"], user["defence"]) <= cap["STATS_CAP"]
    assert max(user[g] for g in GENERALS) <= cap["GENS_CAP"]

matchups = pd.DataFrame({defender_name: {attacker_name: raw_damage(attacker, defender) * neutral_multiplier
                        for attacker_name, attacker in builds.items()}
                        for defender_name, defender in builds.items()})
fig, ax = plt.subplots(figsize=(9, 5))
image = ax.imshow(matchups.to_numpy(), cmap="YlOrRd")
ax.set_xticks(range(len(builds)), builds, rotation=20)
ax.set_yticks(range(len(builds)), builds)
ax.set(xlabel="Defender allocation", ylabel="Attacker allocation", title="Neutral hit damage at equal trained-point budget")
for row in range(len(builds)):
    for column in range(len(builds)):
        ax.text(column, row, f"{matchups.iloc[row, column]:.0f}", ha="center", va="center")
fig.colorbar(image, ax=ax, label="Damage")
plt.tight_layout()
plt.show()


## Damage modifiers and the scope of a raw hit

`computeDamagePacket` applies percentage increases and reductions, static changes, gear/keystone contributions and eligible bloodline effects in their defined order. Increases combine differently depending on their stage; reductions compound, with a minimum derived from damage after increases and the system reduction. Damage increase/reduction effects are universal across damage classifications; seals and bloodline eligibility flags still matter.

The neutral multiplier below includes only the unconditional system increase and reduction. It must not be treated as a full model of modified combat. The reduction cap limits additional reductions relative to the system-reduced boosted damage; it is not simply a floor of 10% of unmodified raw damage. Full interaction analysis should use `computeDamagePacket` and the production combat engine.


In [ ]:
raw = raw_damage(combatant(), combatant())
boosted = raw * (1 + settings["base_increase"] / 100)
system_reduced = boosted * (1 - settings["base_reduction"] / 100)
floor = system_reduced * (1 - settings["reduction_cap"])
illustration = pd.Series({"Raw normalized hit": raw, "After system increase": boosted,
                          "After system reduction": system_reduced,
                          "Additional-reduction floor (no other modifiers)": floor})
ax = illustration.plot.barh(color=["gray", "orange", "teal", "firebrick"])
ax.set(xlabel="Damage", title="System modifiers and the additional-reduction floor")
plt.tight_layout()
plt.show()


## Energy training and progression

Energy capacity is $100 + e(L-1)$ before eligible equipment, bloodline and activated skill capacity effects. Spending Energy instantly adds `stats_per_energy × trainingBoost × trainingMultiplierBoost` points and matching XP to the selected combat stat. Spending is limited to the space below that stat's rank cap; excess requested Energy is not consumed.

Passive recovery uses completed `regen_seconds` ticks. The rates in this section are explicit **scenario inputs**, not estimated player averages. Shared regeneration includes applicable bonuses, sleep and offline recovery, and continues during combat and mastery training. Effective capacity limits idle accumulation; continuously spending can use more than one pool per hour.

Qualifying completion/PvP rewards restore Energy immediately, capped at capacity. Only qualifying terminal quest claims and eligible PvP settlements grant these rewards; arena, sparring, draws and excluded repeat opponents do not. Event rewards use their separate configured value. Timed mastery training uses no Energy and grants no XP; it retains its own time-based gains and daily allowance. Combat mastery growth changes content eligibility rather than the damage formula.


In [ ]:
energy_capacity = 100 + settings["energy_per_level"] * (levels - 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(levels, energy_capacity, label="Base Energy capacity")
axes[0].plot(levels, energy_capacity * settings["stats_per_energy"], label="Full pool → stats, multiplier 1")
axes[0].set(xlabel="Level", ylabel="Energy or stat points", title="Capacity and instant training")
axes[0].legend()

minutes = np.arange(0, 121)
scenario_regen = [1, 10, 100, 400]  # Energy recovered per complete tick; illustrative inputs.
example_capacity = 100 + settings["energy_per_level"] * (50 - 1)
for regen in scenario_regen:
    recovered = np.minimum(example_capacity, np.floor(minutes * 60 / settings["regen_seconds"]) * regen)
    axes[1].step(minutes, recovered, where="post", label=f"{regen} Energy/tick")
axes[1].set(xlabel="Minutes idle from empty", ylabel="Recovered Energy", title="Level 50, base capacity, completed ticks")
axes[1].legend()
plt.tight_layout()
plt.show()

display(pd.DataFrame({"Energy per tick (scenario)": scenario_regen,
                      "Sustained stats/hour, multiplier 1":
                          np.array(scenario_regen) * 3600 / settings["regen_seconds"] * settings["stats_per_energy"]}))
display(pd.Series(settings["energy_rewards"], name="Energy per qualifying claim/settlement").to_frame())


## Interpretation

- Technique power scales raw damage linearly; attack/defence power changes the bounded advantage multiplier.
- Level scales the attacker's base damage independently of invested stats. Defender pool bonuses change hit counts independently of that scaling.
- Offence and Defence govern shared damage power; selected generals retain a substantial and technique-dependent role.
- Mastery enables access to jutsu and equipment without directly adding damage power or XP.
- Energy affects the rate and timing of stat acquisition. It does not enter the damage equation itself.
- Use actual effective stats, pool sizes, configured tuning, content effects and action economy for player-specific or encounter-level conclusions. This notebook does not forecast time to cap, economy costs or PvP win rates without those inputs.
